# Experimentul 5: evaluarea finală pe test

**Trei modele fixate, evaluate o singură dată pe test, cu protocolul oficial (`evaluate_under_noise`) la p = 0, 0.001, 0.01, 0.05.**

| model | ce este | de unde vine |
|---|---|---|
| **A** · VQC baseline | ZZ feature map ×2 + ansatz EfficientSU2-style, 3 straturi, 32 de parametri | Exp. 2, `results/result_exp2/weights/` |
| **B** · VQC simplificat (contribuția noastră) | ZZ feature map ×1, același ansatz, 32 de parametri | Exp. 3, `results/result_exp3/weights/` |
| **C** · regresie logistică | `LogisticRegression(max_iter=1000)` pe aceleași 4 unghiuri | Exp. 4, reconstruită determinist |

Toate trei: același split (`split_indices.npz`), același set de antrenare 1:1 per seed, același `AngleEncoder(4, seed)`, seeds `[0, 1, 2, 3, 4]`.

## Ce face și ce nu face acest notebook
- **Nu antrenează nimic cuantic.** Ponderile VQC sunt încărcate și verificate (AUPRC pe validation la p = 0 identic cu cel salvat). Regresia logistică nu a fost salvată ca obiect, doar coeficienții ei: este reconstruită cu aceeași configurație și verificată (coeficienți, intercept, AUPRC și prag pe validation identice cu Exp. 4).
- Pentru fiecare (model, seed, p): scorurile pe validation la acel p → pragul max-F1 pe validation → același prag aplicat pe test → metricile pe test și Spearman ρ (test cu zgomot vs test fără zgomot). Este corpul funcției `evaluate_under_noise`, rulat câte un p o dată ca să poată fi reluat după întrerupere; echivalența cu funcția oficială este verificată în secțiunea 4.
- Pragurile și AUPRC-ul pe validation trebuie să coincidă cu cele salvate în Exp. 2 / Exp. 3 la **fiecare** p: dovedește că modelul evaluat pe test este exact cel analizat pe validation.
- Rezultatele de pe test **nu** se folosesc pentru nicio alegere. Notebookul doar citește directoarele experimentelor 1–4; scrie numai în `results/result_exp5/`.

## 1. Configurație și imports

**Seeds.** Nu există aleatorietate în această evaluare: ponderile sunt fixe, simularea cu matrici de densitate (`default.mixed`, `shots=None`) este exactă, iar regresia logistică (`lbfgs`) este deterministă. `set_seed` fixează oricum numpy și torch. Seeds 0–4 intervin doar prin setul 1:1 și encoder, exact ca la antrenare.

`FEATURE_MAP_REPS` nu are valoare implicită: circuitul se construiește doar în interiorul `use_feature_map_reps(2)` (model A) sau `use_feature_map_reps(1)` (model B), ca un model să nu poată fi evaluat din greșeală cu circuitul celuilalt.

In [ ]:
import hashlib
import json
import time
from contextlib import contextmanager
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import sklearn
import torch
from IPython.display import Markdown, display
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# ---- protocol comun (identic cu Experimentele 1–4) ----
SPLIT_SEED = 42
N_QUBITS = 4
ANSATZ_REPS = 3               # (3 + 1) * 4 * 2 = 32 parametri
FEATURE_MAP_REPS = None       # setat doar prin use_feature_map_reps(...): A = 2, B = 1
EPOCHS = 30                   # doar pentru verificarea cost ledger-ului (nu se antrenează nimic)
SHOTS_PER_EXECUTION = 1024    # doar pentru cost ledger
NOISE_LEVELS = [0.001, 0.01, 0.05]
NATIVE_GATES = {"RX", "RY", "RZ", "Rot", "Hadamard", "CNOT", "CZ"}
SEEDS = [0, 1, 2, 3, 4]
P_ALL = [0.0, *NOISE_LEVELS]
LR_MAX_ITER = 1000
N_TEST_EXPECTED, N_TEST_FRAUDS_EXPECTED = 56962, 98

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks").is_dir() and (p / "src").is_dir())
RESULTS_DIR = REPO_ROOT / "results" / "result_exp5"
SCORES_DIR = RESULTS_DIR / "scores"                                             # scoruri salvate, pentru reluare fără recalculare
EXP1_DIR = REPO_ROOT / "results" / "result_exp1" / "test1_full_without_noise"   # doar citit
EXP2_DIR = REPO_ROOT / "results" / "result_exp2"                                # doar citit
EXP3_DIR = REPO_ROOT / "results" / "result_exp3"                                # doar citit
EXP4_DIR = REPO_ROOT / "results" / "result_exp4"                                # doar citit
SPLIT_PATH = REPO_ROOT / "split_indices.npz"

MODELS = {   # cele două VQC-uri; regresia logistică (C) este tratată separat
    "A": {"label": "A · VQC baseline (Exp. 2, ZZ ×2)", "short": "A: VQC ZZ ×2", "dir": EXP2_DIR, "reps": 2, "color": "#2a78d6", "marker": "o"},
    "B": {"label": "B · VQC simplificat (Exp. 3, ZZ ×1)", "short": "B: VQC ZZ ×1", "dir": EXP3_DIR, "reps": 1, "color": "#eb6834", "marker": "s"},
}
LR_KEY, LR_LABEL, LR_SHORT, LR_COLOR = "C", "C · regresie logistică (Exp. 4)", "C: regresie logistică", "#3a3a3a"
SEED_DOT_COLOR = "#8a8f98"
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(SPLIT_SEED)
print(f"seeds: {SEEDS}   p: {P_ALL}   modele: A (ZZ ×2), B (ZZ ×1), C (regresie logistică)")
print("pennylane", qml.__version__, "| torch", torch.__version__, "| sklearn", sklearn.__version__, "| numpy", np.__version__)
print("rezultate:", RESULTS_DIR.relative_to(REPO_ROOT))

## 2. Datele și splitul comun

`split_indices.npz` (obligatoriu), verificat față de reconstrucția oficială (Partea 4). Testul trebuie să aibă **56.962 de tranzacții, dintre care 98 de fraude**. Este singurul experiment în care features-urile testului sunt încărcate.

In [ ]:
DATA_PATH = REPO_ROOT / "data" / "creditcard.csv"
for required in [DATA_PATH, SPLIT_PATH]:
    if not required.exists():
        raise FileNotFoundError(f"fișier obligatoriu lipsă: {required}")
df = pd.read_csv(DATA_PATH)
df["Class"] = df["Class"].astype(int)
assert df.shape == (284807, 31) and "Time" in df.columns and df.index.equals(pd.RangeIndex(len(df)))

X_all, y_all = df.drop(columns="Class"), df["Class"].to_numpy()
split = np.load(SPLIT_PATH)
train_idx, val_idx, test_idx = split["train"], split["val"], split["test"]

_trval, _test = train_test_split(np.arange(len(df)), test_size=0.2, stratify=y_all, random_state=SPLIT_SEED)
_train, _val = train_test_split(_trval, test_size=0.25, stratify=y_all[_trval], random_state=SPLIT_SEED)
assert np.array_equal(_train, train_idx) and np.array_equal(_val, val_idx) and np.array_equal(_test, test_idx), "split_indices.npz diferă de splitul oficial"
assert not (set(train_idx) & set(val_idx)) and not (set(train_idx) & set(test_idx)) and not (set(val_idx) & set(test_idx))

X_train, y_train = X_all.loc[train_idx], y_all[train_idx]
X_val, y_val = X_all.loc[val_idx], y_all[val_idx]
X_test, y_test = X_all.loc[test_idx], y_all[test_idx]
assert len(y_test) == N_TEST_EXPECTED, f"testul are {len(y_test)} tranzacții, așteptat {N_TEST_EXPECTED}"
assert int(y_test.sum()) == N_TEST_FRAUDS_EXPECTED, f"testul are {int(y_test.sum())} fraude, așteptat {N_TEST_FRAUDS_EXPECTED}"
assert (int(y_train.sum()), int(y_val.sum())) == (295, 99) and len(y_val) == 56962
del X_all, df, _trval, _test, _train, _val

SPLIT_SHA256 = hashlib.sha256(SPLIT_PATH.read_bytes()).hexdigest()
print(f"split: {SPLIT_PATH.name} (identic cu reconstrucția oficială), sha256 {SPLIT_SHA256[:16]}…")
print(f"train {len(y_train):,} ({int(y_train.sum())} fraude)   validation {len(y_val):,} ({int(y_val.sum())})   "
      f"test {len(y_test):,} ({int(y_test.sum())} fraude, rată {y_test.mean():.3%})")

## 3. Funcțiile din notebookul oficial

`undersample`, `AngleEncoder`, circuitul, `predict`, `choose_threshold`, `evaluate`, `noisy_qnode`, `noisy_predict`, `noise_locations` și `evaluate_under_noise` sunt copiate neschimbate din Exp. 2 / Exp. 3 (= notebookul oficial cu golurile completate). Nu există funcții de antrenare în acest notebook.

In [ ]:
def undersample(X_part, y_part, seed):
    rng = np.random.default_rng(seed)
    fraud = np.flatnonzero(y_part == 1)
    legit = rng.choice(np.flatnonzero(y_part == 0), size=len(fraud), replace=False)
    keep = rng.permutation(np.concatenate([fraud, legit]))
    return X_part.iloc[keep], y_part[keep]


class AngleEncoder:
    raw_columns = ["Time", "Amount"]

    def __init__(self, n_components, seed):
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=n_components, random_state=seed)
        self.angles = MinMaxScaler(feature_range=(0, np.pi))

    def _scaled(self, frame):
        out = frame.copy()
        out[self.raw_columns] = self.scaler.transform(frame[self.raw_columns])
        return out.to_numpy()

    def fit(self, frame):
        self.scaler.fit(frame[self.raw_columns])
        self.angles.fit(self.pca.fit_transform(self._scaled(frame)))
        return self

    def transform(self, frame):
        return np.clip(self.angles.transform(self.pca.transform(self._scaled(frame))), 0, np.pi)


def feature_map(x):
    """ZZ feature map. x holds N_QUBITS angles in [0, pi], or a batch of them (shape (..., N_QUBITS))."""
    for _ in range(FEATURE_MAP_REPS):
        for i in range(N_QUBITS):
            qml.Hadamard(wires=i)
            qml.RZ(2 * x[..., i], wires=i)
        for i in range(N_QUBITS):
            for j in range(i + 1, N_QUBITS):
                qml.CNOT(wires=[i, j])
                qml.RZ(2 * (np.pi - x[..., i]) * (np.pi - x[..., j]), wires=j)
                qml.CNOT(wires=[i, j])


def ansatz(weights):
    """EfficientSU2-style ansatz; weights has shape (ANSATZ_REPS + 1, N_QUBITS, 2)."""
    for layer in range(ANSATZ_REPS + 1):
        for i in range(N_QUBITS):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        if layer < ANSATZ_REPS:
            for i in range(N_QUBITS - 1):
                qml.CNOT(wires=[i, i + 1])


def vqc(x, weights):
    feature_map(x)
    ansatz(weights)
    return qml.expval(qml.PauliZ(0))


device = qml.device("default.qubit", wires=N_QUBITS)   # shots=None: valori de așteptare exacte
circuit = qml.QNode(vqc, device, interface="torch", diff_method="backprop")


def fraud_probability(x, weights):
    return (1 - circuit(x, weights)) / 2


def predict(Z, weights, model=None, chunk=20000):
    model = model or fraud_probability
    X_t = torch.tensor(Z)
    with torch.no_grad():
        return torch.cat([model(X_t[k:k + chunk], weights) for k in range(0, len(X_t), chunk)]).numpy()


def choose_threshold(y_true, scores):
    """The threshold that maximises F1 on (y_true, scores)."""
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
    return thresholds[np.argmax(f1[:-1])]


def evaluate(y_true, scores, threshold):
    predicted = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    return {
        "AUPRC": average_precision_score(y_true, scores),
        "AUROC": roc_auc_score(y_true, scores),
        "precision": precision_score(y_true, predicted, zero_division=0),
        "recall": recall_score(y_true, predicted, zero_division=0),
        "F1": f1_score(y_true, predicted, zero_division=0),
        "MCC": matthews_corrcoef(y_true, predicted),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
    }


def noisy_qnode(circuit_fn, p, n_wires=N_QUBITS):
    """QNode of circuit_fn under the challenge's noise model: native gates, then a depolarizing channel of
    probability p after every gate, on each of its qubits. Differentiable with backprop."""
    noisy_device = qml.noise.insert(qml.device("default.mixed", wires=n_wires), qml.DepolarizingChannel, p, position="all")
    qnode = qml.QNode(circuit_fn, noisy_device, interface="torch", diff_method="backprop")
    return qml.transforms.decompose(qnode, gate_set=NATIVE_GATES)


def noisy_predict(Z, weights, p, circuit_fn=vqc, readout=lambda expval: (1 - expval) / 2, n_wires=N_QUBITS, chunk=2000):
    """Fraud scores of the transactions Z when circuit_fn runs under depolarizing noise p."""
    noisy_circuit = noisy_qnode(circuit_fn, p, n_wires)
    X_t = torch.tensor(Z)
    with torch.no_grad():
        return torch.cat([readout(noisy_circuit(X_t[k:k + chunk], weights)) for k in range(0, len(X_t), chunk)]).numpy()


def noise_locations(circuit_fn, *args, n_wires=N_QUBITS):
    """Number of depolarizing channels the noise model inserts: one per qubit of every native gate."""
    qnode = qml.transforms.decompose(qml.QNode(circuit_fn, qml.device("default.qubit", wires=n_wires)), gate_set=NATIVE_GATES)
    return sum(size * count for size, count in qml.specs(qnode)(*args)["resources"].gate_sizes.items())


def evaluate_under_noise(circuit_fn, weights, Z_valid, Z_test_part, y_valid=y_val, y_test_part=y_test,
                         readout=lambda expval: (1 - expval) / 2, noise_levels=NOISE_LEVELS, n_wires=N_QUBITS):
    clean = qml.QNode(circuit_fn, qml.device("default.qubit", wires=n_wires), interface="torch")
    with torch.no_grad():
        clean_val = readout(clean(torch.tensor(Z_valid), weights)).numpy()
        clean_test = readout(clean(torch.tensor(Z_test_part), weights)).numpy()
    rows = []
    for p in [0.0, *noise_levels]:
        if p == 0.0:
            val_scores, test_scores = clean_val, clean_test
        else:
            val_scores = noisy_predict(Z_valid, weights, p, circuit_fn, readout, n_wires)
            test_scores = noisy_predict(Z_test_part, weights, p, circuit_fn, readout, n_wires)
        threshold = choose_threshold(y_valid, val_scores)
        rows.append({"p": p, **evaluate(y_test_part, test_scores, threshold), "Spearman rho": spearmanr(clean_test, test_scores).statistic, "threshold": threshold})
    return pd.DataFrame(rows).set_index("p")


# ---- utilitare (ca în Experimentele 2–4) ----
@contextmanager
def use_feature_map_reps(reps):
    """Circuitul modelului A (reps = 2) sau B (reps = 1); FEATURE_MAP_REPS revine la None la ieșire."""
    global FEATURE_MAP_REPS
    saved, FEATURE_MAP_REPS = FEATURE_MAP_REPS, reps
    try:
        yield
    finally:
        FEATURE_MAP_REPS = saved


def protocol_row(y_valid, val_scores, y_test_part, test_scores, clean_test):
    """Un rând din evaluate_under_noise: prag max-F1 pe validation, aplicat neschimbat pe test."""
    threshold = choose_threshold(y_valid, val_scores)
    return {**evaluate(y_test_part, test_scores, threshold), "Spearman rho": spearmanr(clean_test, test_scores).statistic, "threshold": threshold}


def load_jsonl(path, required=False):
    """Înregistrările din path. Un fișier obligatoriu lipsă oprește execuția, în loc să întoarcă o listă goală."""
    if not path.exists():
        if required:
            raise FileNotFoundError(f"fișier obligatoriu lipsă: {path}")
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def to_builtin(value):
    if isinstance(value, dict):
        return {k: to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(to_builtin(record), ensure_ascii=False) + "\n")


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def mean_std(values, digits=3):
    values = np.asarray(values, dtype=float)
    return f"{values.mean():.{digits}f} ± {values.std(ddof=1):.{digits}f}"


# ✅ Checks din notebook: 9.1 (pragul maximizează F1) și 10.1 (canalul depolarizant)
rng_check = np.random.default_rng(3)
y_c = rng_check.integers(0, 2, 300)
s_c = np.clip(0.3 * y_c + rng_check.random(300), 0, 1)
best_f1 = max(f1_score(y_c, (s_c >= t).astype(int)) for t in np.unique(s_c))
assert np.isclose(f1_score(y_c, (s_c >= choose_threshold(y_c, s_c)).astype(int)), best_f1)


@qml.qnode(qml.device("default.mixed", wires=1))
def noisy_ry(theta, p):
    qml.RY(theta, wires=0)
    qml.DepolarizingChannel(p, wires=0)
    return qml.expval(qml.PauliZ(0))


for theta, p in [(0.3, 0.0), (1.0, 0.1), (2.5, 0.3), (0.7, 0.75)]:
    assert np.isclose(noisy_ry(theta, p), (1 - 4 * p / 3) * np.cos(theta))
print("✅ Checks 9.1, 10.1: trecute")

## 4. Modelele fixate: încărcare și verificare (fără reantrenare)

Pentru fiecare seed se reconstruiește determinist din training setul 1:1 (`undersample`) și encoderul (`AngleEncoder(4, seed)` fitat pe el), exact ca în Exp. 2–4. Testul este doar **transformat** de encoderul fitat pe train.

Verificări, toate obligatorii (orice abatere oprește notebookul):
1. **VQC A și B**: ponderile salvate, pe unghiurile de validation reconstruite, reproduc AUPRC-ul salvat la p = 0 (toleranță 1e-12). Asta confirmă simultan ponderile, encoderul și circuitul.
2. **Regresia logistică**: reantrenată cu configurația fixată (`max_iter=1000`, restul implicit, fără nicio căutare) pe același set 1:1; coeficienții și interceptul trebuie să fie identici cu cei salvați în Exp. 4, iar AUPRC-ul și pragul pe validation la fel.
3. Amprentele sha256 ale fișierelor de ponderi și ale splitului se scriu în `config.json`; o reluare cu alte fișiere se oprește.

In [ ]:
saved_val = {key: {(r["seed"], r["p"]): r for r in load_jsonl(m["dir"] / "val_runs.jsonl", required=True)} for key, m in MODELS.items()}
lr_saved = {r["seed"]: r for r in load_jsonl(EXP4_DIR / "val_runs.jsonl", required=True)}
for key in MODELS:
    assert set(saved_val[key]) >= {(s, p) for s in SEEDS for p in P_ALL}, f"model {key}: lipsesc rezultate pe validation"
assert set(lr_saved) >= set(SEEDS)

inputs, weights, lr_models, check_rows, weight_hashes = {}, {key: {} for key in MODELS}, {}, [], {}
for seed in SEEDS:
    X_s, y_s = undersample(X_train, y_train, seed)
    encoder = AngleEncoder(N_QUBITS, seed).fit(X_s)   # fitat doar pe setul 1:1 din train
    Z_tr, Z_val, Z_test = encoder.transform(X_s), encoder.transform(X_val), encoder.transform(X_test)
    assert Z_tr.shape == (590, N_QUBITS) and int(y_s.sum()) == 295 and Z_test.shape == (N_TEST_EXPECTED, N_QUBITS)
    assert Z_test.min() >= 0 and Z_test.max() <= np.pi
    inputs[seed] = {"Z_tr": Z_tr, "y_tr": y_s, "Z_val": Z_val, "Z_test": Z_test}
    row = {"seed": seed}

    for key, m in MODELS.items():
        path = m["dir"] / "weights" / f"seed_{seed}.pt"
        saved = torch.load(path, weights_only=True)
        assert saved["weights"].shape == (ANSATZ_REPS + 1, N_QUBITS, 2)
        weight_hashes[f"{key}/seed_{seed}.pt"] = sha256(path)
        with use_feature_map_reps(m["reps"]):
            ap = average_precision_score(y_val, predict(Z_val, saved["weights"]))
        ref = saved_val[key][(seed, 0.0)]["AUPRC"]
        assert np.isclose(ap, ref, rtol=0, atol=1e-12) and np.isclose(ap, saved["best_val_ap"], rtol=0, atol=1e-12), \
            f"model {key}, seed {seed}: AUPRC {ap:.12f} vs salvat {ref:.12f}"
        weights[key][seed] = saved["weights"]
        row.update({f"{key}: epoca aleasă": saved["best_epoch"], f"{key}: val AUPRC recalculat": ap, f"{key}: val AUPRC salvat": ref})

    set_seed(seed)
    lr_model = LogisticRegression(max_iter=LR_MAX_ITER).fit(Z_tr, y_s)
    ref = lr_saved[seed]
    lr_val = lr_model.predict_proba(Z_val)[:, 1]
    lr_ap, lr_thr = average_precision_score(y_val, lr_val), choose_threshold(y_val, lr_val)
    assert np.allclose(lr_model.coef_[0], ref["coeficienți"], rtol=0, atol=1e-10), f"seed {seed}: coeficienții LR diferă de Exp. 4"
    assert np.isclose(lr_model.intercept_[0], ref["intercept"], rtol=0, atol=1e-10), f"seed {seed}: interceptul LR diferă de Exp. 4"
    assert np.isclose(lr_ap, ref["AUPRC"], rtol=0, atol=1e-12) and np.isclose(lr_thr, ref["threshold"], rtol=0, atol=1e-12)
    lr_models[seed] = lr_model
    row.update({"C: |Δ coeficienți| max": float(np.abs(lr_model.coef_[0] - np.array(ref["coeficienți"])).max()),
                "C: val AUPRC recalculat": lr_ap, "C: val AUPRC salvat": ref["AUPRC"]})
    check_rows.append(row)
    print(f"seed {seed}: A val AUPRC {row['A: val AUPRC recalculat']:.4f} ✓   B {row['B: val AUPRC recalculat']:.4f} ✓   "
          f"C {lr_ap:.4f} ✓ (coeficienți identici cu Exp. 4)")

model_check = pd.DataFrame(check_rows).set_index("seed")
print("✅ modele fixate: ponderile VQC și regresia logistică reproduc exact rezultatele salvate pe validation")
display(model_check.round(6))

### Echivalența cu funcția oficială și sanity check pe simulator

- `evaluate_under_noise` oficial vs `protocol_row` (folosit mai jos, câte un p o dată) pe un subset stratificat (toate fraudele + 1.500 de tranzacții legitime din validation și din test), model B, seed 0, p = 0 și 0.01: rezultatele trebuie să fie identice.
- `default.mixed` la p = 0 dă aceleași scoruri ca state-vector (diferență ~1e-7, precizie numerică).

In [ ]:
sub_val = np.r_[np.flatnonzero(y_val == 1), np.flatnonzero(y_val == 0)[:1500]]
sub_test = np.r_[np.flatnonzero(y_test == 1), np.flatnonzero(y_test == 0)[:1500]]
w0, Zv0, Zt0 = weights["B"][SEEDS[0]], inputs[SEEDS[0]]["Z_val"][sub_val], inputs[SEEDS[0]]["Z_test"][sub_test]
with use_feature_map_reps(MODELS["B"]["reps"]):
    official = evaluate_under_noise(vqc, w0, Zv0, Zt0, y_valid=y_val[sub_val], y_test_part=y_test[sub_test], noise_levels=[0.01])
    clean_v, clean_t = predict(Zv0, w0), predict(Zt0, w0)
    mine = pd.DataFrame([
        {"p": 0.0, **protocol_row(y_val[sub_val], clean_v, y_test[sub_test], clean_t, clean_t)},
        {"p": 0.01, **protocol_row(y_val[sub_val], noisy_predict(Zv0, w0, 0.01), y_test[sub_test], noisy_predict(Zt0, w0, 0.01), clean_t)},
    ]).set_index("p")
    sanity = np.abs(noisy_predict(Zv0, w0, 0.0) - clean_v).max()
assert np.allclose(official[mine.columns].to_numpy(dtype=float), mine.to_numpy(dtype=float), rtol=0, atol=1e-10), "protocol_row diferă de evaluate_under_noise"
assert sanity < 1e-6, sanity
print(f"✅ protocol_row = evaluate_under_noise oficial (subset {len(sub_val):,} val / {len(sub_test):,} test, p = 0 și 0.01)")
print(f"✅ default.mixed la p = 0 vs default.qubit: diferență maximă {sanity:.1e}")

## 5. Evaluarea finală pe test

Pentru fiecare model VQC, seed și p (p = 0 primul: scorurile lui pe test sunt referința fără zgomot pentru Spearman ρ):
1. scorurile pe validation și pe test la același p (`predict` la p = 0, `noisy_predict` cu modelul oficial de zgomot în rest);
2. pragul max-F1 pe validation; **verificare**: pragul și AUPRC-ul pe validation coincid cu cele salvate în Exp. 2 / Exp. 3 la același p;
3. același prag, neschimbat, pe test → AUPRC, AUROC, precision, recall, F1, MCC, TP / FP / FN / TN, Spearman ρ (test cu zgomot vs test fără zgomot).

Regresia logistică: scoruri pe test, pragul ei ales pe validation în Exp. 4 (reverificat mai sus), aplicat neschimbat.

**Reluare după întrerupere.** Scorurile fiecărei treceri se salvează în `scores/` (scriere atomică), iar fiecare rezultat terminat se adaugă imediat în `test_runs.jsonl`. La o nouă rulare, perechile deja terminate sunt sărite, iar scorurile deja calculate sunt încărcate, nu recalculate.

**Durată estimată:** 2 modele × 5 seeds × 3 niveluri cu zgomot × 2 treceri (validation + test) = 60 de treceri zgomotoase, ~25–35 min pe laptop (după timpii din Exp. 2 și Exp. 3).

In [ ]:
config = {
    "experiment": "final_test_evaluation",
    "models": {key: {"label": m["label"], "feature_map_reps": m["reps"], "ansatz_reps": ANSATZ_REPS, "n_params": 32,
                     "weights": str(m["dir"].relative_to(REPO_ROOT) / "weights")} for key, m in MODELS.items()},
    "classical_reference": f"LogisticRegression(max_iter={LR_MAX_ITER}), rebuilt deterministically, coefficients verified against Exp. 4",
    "seeds": SEEDS, "noise_levels": P_ALL, "split": "split_indices.npz (verified against official reconstruction)",
    "split_sha256": SPLIT_SHA256, "weights_sha256": weight_hashes,
    "inputs": "undersample(X_train, y_train, seed) 1:1 + AngleEncoder(4, seed) fitted on the 1:1 set",
    "noise_model": "noisy_qnode: NATIVE_GATES decomposition, DepolarizingChannel(p) after every gate on every wire, default.mixed",
    "protocol": "evaluate_under_noise per (seed, p): threshold = max F1 on validation at the same p, applied unchanged to test",
    "n_test": int(len(y_test)), "n_test_frauds": int(y_test.sum()), "shots_per_execution": SHOTS_PER_EXECUTION,
    "versions": {"pennylane": qml.__version__, "torch": torch.__version__, "sklearn": sklearn.__version__, "numpy": np.__version__},
}
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SCORES_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH, TEST_RUNS_PATH = RESULTS_DIR / "config.json", RESULTS_DIR / "test_runs.jsonl"
if CONFIG_PATH.exists():
    previous = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    current = json.loads(json.dumps(to_builtin(config)))
    if {k: v for k, v in previous.items() if k != "versions"} != {k: v for k, v in current.items() if k != "versions"}:
        raise RuntimeError(f"{CONFIG_PATH} conține altă configurație (modele, ponderi sau split diferite). Nu suprascriu.")
else:
    CONFIG_PATH.write_text(json.dumps(to_builtin(config), indent=2, ensure_ascii=False), encoding="utf-8")


def get_scores(key, seed, p):
    """Scorurile (validation, test) ale modelului key la zgomotul p: încărcate din scores/ sau calculate și salvate."""
    path = SCORES_DIR / f"{key}_seed{seed}_p{p}.npz"
    if path.exists():
        saved = np.load(path)
        return saved["val"], saved["test"], float(saved["seconds"])
    if key == LR_KEY:
        start = time.perf_counter()
        val, test = (lr_models[seed].predict_proba(inputs[seed][part])[:, 1] for part in ["Z_val", "Z_test"])
    else:
        w, Z_v, Z_t = weights[key][seed], inputs[seed]["Z_val"], inputs[seed]["Z_test"]
        start = time.perf_counter()
        with use_feature_map_reps(MODELS[key]["reps"]):
            val, test = (predict(Z_v, w), predict(Z_t, w)) if p == 0.0 else (noisy_predict(Z_v, w, p), noisy_predict(Z_t, w, p))
    seconds = time.perf_counter() - start
    tmp = path.with_name(path.stem + ".tmp.npz")
    np.savez(tmp, val=val, test=test, seconds=seconds)
    tmp.replace(path)   # scriere atomică: o întrerupere nu lasă un fișier incomplet
    return val, test, seconds


test_done = {(r["model"], r["seed"], r["p"]) for r in load_jsonl(TEST_RUNS_PATH)}
todo = [(key, s, p) for key in MODELS for s in SEEDS for p in P_ALL] + [(LR_KEY, s, 0.0) for s in SEEDS]
print(f"evaluări de făcut: {sum(t not in test_done for t in todo)} din {len(todo)}")

for key in MODELS:
    for seed in SEEDS:
        for p in P_ALL:
            if (key, seed, p) in test_done:
                continue
            val_scores, test_scores, seconds = get_scores(key, seed, p)
            clean_test = test_scores if p == 0.0 else get_scores(key, seed, 0.0)[1]
            row = protocol_row(y_val, val_scores, y_test, test_scores, clean_test)
            val_ap, ref = average_precision_score(y_val, val_scores), saved_val[key][(seed, p)]
            assert np.isclose(row["threshold"], ref["threshold"], rtol=0, atol=1e-9), f"{key}, seed {seed}, p = {p}: pragul diferă de validation"
            assert np.isclose(val_ap, ref["AUPRC"], rtol=0, atol=1e-9), f"{key}, seed {seed}, p = {p}: AUPRC pe validation diferă"
            append_jsonl(TEST_RUNS_PATH, {"model": key, "seed": seed, "p": p, **row, "threshold": float(row["threshold"]),
                                          "val AUPRC": val_ap, "timp scorare val + test [s]": seconds})
            test_done.add((key, seed, p))
            print(f"{key} seed {seed}  p = {p:<5}  test AUPRC {row['AUPRC']:.3f}  F1 {row['F1']:.3f}  MCC {row['MCC']:.3f}  "
                  f"rho {row['Spearman rho']:.4f}  prag {row['threshold']:.3f} (= validation ✓)  ({seconds:.0f} s)")

for seed in SEEDS:   # regresia logistică: clasică, aceleași scoruri la orice p
    if (LR_KEY, seed, 0.0) in test_done:
        continue
    val_scores, test_scores, seconds = get_scores(LR_KEY, seed, 0.0)
    threshold = choose_threshold(y_val, val_scores)
    assert np.isclose(threshold, lr_saved[seed]["threshold"], rtol=0, atol=1e-12)
    metrics = evaluate(y_test, test_scores, threshold)
    append_jsonl(TEST_RUNS_PATH, {"model": LR_KEY, "seed": seed, "p": 0.0, **metrics, "Spearman rho": None, "threshold": float(threshold),
                                  "val AUPRC": average_precision_score(y_val, val_scores), "timp scorare val + test [s]": seconds})
    test_done.add((LR_KEY, seed, 0.0))
    print(f"C seed {seed}  test AUPRC {metrics['AUPRC']:.3f}  F1 {metrics['F1']:.3f}  MCC {metrics['MCC']:.3f}  prag {threshold:.3f} (= Exp. 4 ✓)")
print("✅ evaluarea pe test completă")

## 6. Rezultate pe test

Medie ± abatere standard (ddof = 1) pe cele 5 seeds. Regresia logistică nu este afectată de zgomotul cuantic: are aceeași valoare la fiecare p și apare ca reper. Spearman ρ nu are sens pentru ea (ar fi 1 prin construcție).

In [ ]:
METRICS = ["AUPRC", "AUROC", "Spearman rho", "precision", "recall", "F1", "MCC"]
COUNTS = ["TP", "FP", "FN", "TN"]
LABELS = {**{k: m["label"] for k, m in MODELS.items()}, LR_KEY: LR_LABEL}
SHORT = {**{k: m["short"] for k, m in MODELS.items()}, LR_KEY: LR_SHORT}

records = load_jsonl(TEST_RUNS_PATH, required=True)
keys = [(r["model"], r["seed"], r["p"]) for r in records]
assert len(keys) == len(set(keys)), "rezultate duplicate în test_runs.jsonl"
assert set(keys) == set(todo), f"rezultate lipsă: {sorted(set(todo) - set(keys))}"
raw = pd.DataFrame(records)
lr_rows = raw[raw["model"] == LR_KEY]
test = pd.concat([raw[raw["model"] != LR_KEY], *[lr_rows.assign(p=p) for p in P_ALL]], ignore_index=True)   # LR repetată la fiecare p
test["Spearman rho"] = test["Spearman rho"].astype(float)
test = test.sort_values(["model", "p", "seed"]).reset_index(drop=True)


def part(model, p, column):
    return test[(test["model"] == model) & (test["p"] == p)].set_index("seed").loc[SEEDS, column].astype(float)


detail_cols = ["model", "seed", "p", *METRICS, *COUNTS, "threshold", "val AUPRC"]
raw[detail_cols].to_csv(RESULTS_DIR / "test_detailed.csv", index=False)

summary_rows, numeric_rows = [], []
for model in ["A", "B", LR_KEY]:
    for p in P_ALL:
        summary = {"model": LABELS[model], "p": p}
        for m in METRICS:
            values = part(model, p, m)
            if model == LR_KEY and m == "Spearman rho":
                summary[m] = "n/a (clasic)"
                continue
            summary[m] = mean_std(values, 4 if m == "Spearman rho" else 3)
            numeric_rows.append({"model": model, "p": p, "metrică": m, "medie": values.mean(), "std": values.std(ddof=1),
                                 "min": values.min(), "max": values.max()})
        for c in COUNTS:
            summary[c] = mean_std(part(model, p, c), 1)
        summary_rows.append(summary)
test_summary = pd.DataFrame(summary_rows).set_index(["model", "p"])
test_numeric = pd.DataFrame(numeric_rows)
test_summary.to_csv(RESULTS_DIR / "test_summary.csv")
test_numeric.to_csv(RESULTS_DIR / "test_summary_numeric.csv", index=False)

print("Tabel detaliat pe test (câte un rând per model, seed, p; LR o dată per seed)")
display(raw[detail_cols].round(4))
print("Rezumat pe test, medie ± std pe 5 seeds")
display(test_summary)

## 7. Comparațiile dintre modele

Toate diferențele sunt **pereche**: același seed = același set 1:1, același encoder; la A vs B și aceleași ponderi inițiale. Pentru fiecare diferență: media ± std pe seeds și în câte din cele 5 seeds semnul este favorabil. Toate seeds sunt raportate, inclusiv cele nefavorabile.

In [ ]:
def paired_table(first, second, p_list=P_ALL, metrics=("AUPRC", "AUROC", "Spearman rho", "F1", "MCC")):
    rows = []
    for p in p_list:
        for m in metrics:
            if LR_KEY in (first, second) and m == "Spearman rho":
                continue
            a, b = part(first, p, m), part(second, p, m)
            d, digits = b - a, 4 if m == "Spearman rho" else 3
            rows.append({"p": p, "metrică": m, SHORT[first]: mean_std(a, digits), SHORT[second]: mean_std(b, digits),
                         f"Δ pereche ({second} − {first})": f"{d.mean():+.{digits}f} ± {d.std(ddof=1):.{digits}f}",
                         f"seeds cu {second} > {first}": f"{int((d > 0).sum())} / {len(d)}",
                         "Δ min": round(d.min(), digits), "Δ max": round(d.max(), digits)})
    return pd.DataFrame(rows).set_index(["p", "metrică"])


cmp_ab = paired_table("A", "B")
cmp_lr_b = paired_table(LR_KEY, "B")
cmp_lr_a = paired_table(LR_KEY, "A")
cmp_ab.to_csv(RESULTS_DIR / "comparison_A_vs_B.csv")
cmp_lr_b.to_csv(RESULTS_DIR / "comparison_B_vs_LR.csv")
cmp_lr_a.to_csv(RESULTS_DIR / "comparison_A_vs_LR.csv")
print("1. Baseline VQC (A) vs VQC simplificat (B), test")
display(cmp_ab)
print("2. Regresie logistică (C) vs VQC simplificat (B), test (Δ < 0: regresia logistică mai bună)")
display(cmp_lr_b)
print("   Regresie logistică (C) vs baseline VQC (A), test")
display(cmp_lr_a.loc[[(p, "AUPRC") for p in P_ALL]])

per_seed = []
for p in P_ALL:
    for seed in SEEDS:
        a, b, c = (part(k, p, "AUPRC")[seed] for k in ["A", "B", LR_KEY])
        per_seed.append({"p": p, "seed": seed, "AUPRC A": a, "AUPRC B": b, "AUPRC C (LR)": c,
                         "Δ B − A": b - a, "Δ B − C": b - c, "Δ A − C": a - c})
per_seed = pd.DataFrame(per_seed)
per_seed.to_csv(RESULTS_DIR / "paired_auprc_per_seed.csv", index=False)
print("3. AUPRC pe test per seed și diferențele pereche")
display(per_seed.set_index(["p", "seed"]).round(4))

In [ ]:
# 5. Degradarea AUPRC față de propriul p = 0 și 6. stabilitatea între seeds
degradation = []
for model in MODELS:
    base = part(model, 0.0, "AUPRC")
    for p in P_ALL:
        cur = part(model, p, "AUPRC")
        for seed in SEEDS:
            degradation.append({"model": model, "p": p, "seed": seed, "AUPRC": cur[seed], "Δ față de p = 0": cur[seed] - base[seed],
                                "Δ relativ [%]": 100 * (cur[seed] - base[seed]) / base[seed]})
degradation = pd.DataFrame(degradation)
degradation.to_csv(RESULTS_DIR / "degradation_vs_p0.csv", index=False)
deg_summary = degradation.groupby(["model", "p"]).agg(**{
    "Δ AUPRC față de p = 0": ("Δ față de p = 0", lambda v: f"{v.mean():+.3f} ± {v.std(ddof=1):.3f}"),
    "Δ relativ [%]": ("Δ relativ [%]", lambda v: f"{v.mean():+.1f} ± {v.std(ddof=1):.1f}"),
    "cea mai mare pierdere (seed)": ("Δ față de p = 0", "min"),
})
deg_summary.to_csv(RESULTS_DIR / "degradation_summary.csv")
print("5. Degradarea AUPRC pe test față de p = 0 al aceluiași model")
display(deg_summary.round(3))

stability = []
for model in ["A", "B", LR_KEY]:
    for p in P_ALL:
        for m in ["AUPRC", "Spearman rho"]:
            if model == LR_KEY and m == "Spearman rho":
                continue
            v = part(model, p, m)
            stability.append({"model": model, "p": p, "metrică": m, "medie": v.mean(), "std": v.std(ddof=1), "min": v.min(), "max": v.max(),
                              "interval (max − min)": v.max() - v.min(), "CV [%]": 100 * v.std(ddof=1) / v.mean(),
                              "seed cel mai slab": int(v.idxmin())})
stability = pd.DataFrame(stability).set_index(["model", "p", "metrică"])
stability.to_csv(RESULTS_DIR / "stability_across_seeds.csv")
print("6. Stabilitatea între seeds (test)")
display(stability.round(4))

### Se confirmă rezultatele de pe validation?

AUPRC pe validation (salvat în Exp. 2–4, recalculat și verificat aici) vs AUPRC pe test, pentru aceleași modele și praguri. Precision / recall / F1 / MCC pe validation erau optimiste (prag ales pe același set); pe test pragul vine din validation, deci sunt estimări oneste. Verificăm și dacă **clasamentul** B > A se păstrează seed cu seed.

In [ ]:
val_test = []
for model in ["A", "B", LR_KEY]:
    for p in P_ALL:
        v_ap, t_ap = part(model, p, "val AUPRC"), part(model, p, "AUPRC")
        if model == LR_KEY:
            v_f1 = pd.Series({s: lr_saved[s]["F1"] for s in SEEDS})
        else:
            v_f1 = pd.Series({s: saved_val[model][(s, p)]["F1"] for s in SEEDS})
        t_f1 = part(model, p, "F1")
        val_test.append({"model": model, "p": p, "AUPRC validation": mean_std(v_ap), "AUPRC test": mean_std(t_ap),
                         "Δ AUPRC (test − val)": f"{(t_ap - v_ap).mean():+.3f}", "F1 validation (optimist)": mean_std(v_f1),
                         "F1 test": mean_std(t_f1), "Δ F1 (test − val)": f"{(t_f1 - v_f1).mean():+.3f}"})
val_test = pd.DataFrame(val_test).set_index(["model", "p"])

ranking = []
for p in P_ALL:
    d_val = part("B", p, "val AUPRC") - part("A", p, "val AUPRC")
    d_test = part("B", p, "AUPRC") - part("A", p, "AUPRC")
    d_val_lr = part("B", p, "val AUPRC") - part(LR_KEY, p, "val AUPRC")
    d_test_lr = part("B", p, "AUPRC") - part(LR_KEY, p, "AUPRC")
    ranking.append({"p": p, "Δ AUPRC B − A validation": f"{d_val.mean():+.3f} ({int((d_val > 0).sum())}/5 seeds B > A)",
                    "Δ AUPRC B − A test": f"{d_test.mean():+.3f} ({int((d_test > 0).sum())}/5 seeds B > A)",
                    "Δ AUPRC B − C validation": f"{d_val_lr.mean():+.3f} ({int((d_val_lr > 0).sum())}/5 seeds B > C)",
                    "Δ AUPRC B − C test": f"{d_test_lr.mean():+.3f} ({int((d_test_lr > 0).sum())}/5 seeds B > C)"})
ranking = pd.DataFrame(ranking).set_index("p")
val_test.to_csv(RESULTS_DIR / "validation_vs_test.csv")
ranking.to_csv(RESULTS_DIR / "ranking_validation_vs_test.csv")
print("AUPRC și F1: validation vs test (medie ± std pe 5 seeds)")
display(val_test)
print("Clasamentul pereche: se păstrează pe test?")
display(ranking)

### Grafice

Punctele gri sunt seeds individuale, marcajele colorate cu bare sunt media ± std. Banda gri închis este regresia logistică (medie ± std), aceeași la orice p.

In [ ]:
xpos = np.arange(len(P_ALL))
xlabels = ["0", *[str(p) for p in NOISE_LEVELS]]
OFFSETS = {"A": -0.07, "B": 0.07}


def vqc_series(ax, col):
    for key, m in MODELS.items():
        groups = [part(key, p, col).to_numpy() for p in P_ALL]
        for i, g in enumerate(groups):
            ax.scatter(np.full(len(g), xpos[i] + OFFSETS[key]), g, s=18, color=SEED_DOT_COLOR, alpha=0.6, zorder=2)
        ax.errorbar(xpos + OFFSETS[key], [g.mean() for g in groups], yerr=[g.std(ddof=1) for g in groups], color=m["color"],
                    marker=m["marker"], markersize=8, linewidth=2, capsize=4, label=m["label"], zorder=3)


def lr_band(ax, col):
    v = part(LR_KEY, 0.0, col)
    ax.axhspan(v.mean() - v.std(ddof=1), v.mean() + v.std(ddof=1), color=LR_COLOR, alpha=0.10, zorder=1)
    ax.axhline(v.mean(), color=LR_COLOR, linestyle="--", linewidth=2, label=f"{LR_LABEL}, ± std", zorder=2)


# 1, 2, 4: AUPRC, F1 și Spearman rho vs p
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
for ax, col, title in zip(axes, ["AUPRC", "F1", "Spearman rho"],
                          ["AUPRC pe test vs p", "F1 pe test (prag ales pe validation la același p)", "Spearman ρ pe test, cu zgomot vs fără zgomot"]):
    if col != "Spearman rho":
        lr_band(ax, col)
    vqc_series(ax, col)
    ax.set_xticks(xpos, xlabels)
    ax.set_xlabel("probabilitate depolarizantă p per poartă (doar VQC)")
    ax.set_title(title)
axes[0].legend(loc="lower left", fontsize=8)
fig.suptitle("Evaluarea finală pe TEST (56.962 tranzacții, 98 fraude) · medie ± std pe 5 seeds (gri: seeds)", fontsize=11)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "test_metrics_vs_p.png", dpi=130)
plt.show()

# 3: diferențe pereche de AUPRC per seed
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True)
for ax, (col, title, color) in zip(axes, [("Δ B − A", "Δ AUPRC: B (ZZ ×1) − A (ZZ ×2)", MODELS["B"]["color"]),
                                          ("Δ B − C", "Δ AUPRC: B (ZZ ×1) − regresie logistică", MODELS["B"]["color"]),
                                          ("Δ A − C", "Δ AUPRC: A (ZZ ×2) − regresie logistică", MODELS["A"]["color"])]):
    for seed in SEEDS:
        d = per_seed[per_seed["seed"] == seed].set_index("p").loc[P_ALL, col]
        ax.plot(xpos, d, color=SEED_DOT_COLOR, marker="o", markersize=5, linewidth=1, alpha=0.8, zorder=2)
        ax.annotate(f"s{seed}", (xpos[-1], d.iloc[-1]), textcoords="offset points", xytext=(6, 0), fontsize=8, color="0.35", va="center")
    ax.plot(xpos, per_seed.groupby("p")[col].mean().loc[P_ALL], color=color, marker="s", markersize=8, linewidth=2, label="medie pe 5 seeds", zorder=3)
    ax.axhline(0, color="0.3", linestyle="--", linewidth=1)
    ax.set_xticks(xpos, xlabels)
    ax.set_xlabel("probabilitate depolarizantă p per poartă")
    ax.set_title(title)
    ax.legend(loc="lower left", fontsize=8)
axes[0].set_ylabel("Δ AUPRC pe test (> 0: primul model mai bun)")
fig.suptitle("Diferențe pereche pe seed, test (gri: câte o linie per seed)", fontsize=11)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "paired_auprc_differences.png", dpi=130)
plt.show()

# 5 și 6: degradarea față de p = 0 și stabilitatea per seed la p = 0.01 / 0.05
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
ax = axes[0]
for key, m in MODELS.items():
    part_d = degradation[degradation["model"] == key]
    for seed in SEEDS:
        d = part_d[part_d["seed"] == seed].set_index("p").loc[P_ALL, "Δ față de p = 0"]
        ax.plot(xpos + OFFSETS[key], d, color=m["color"], alpha=0.25, linewidth=1, zorder=2)
    g = part_d.groupby("p")["Δ față de p = 0"]
    ax.errorbar(xpos + OFFSETS[key], g.mean().loc[P_ALL], yerr=g.std(ddof=1).loc[P_ALL], color=m["color"], marker=m["marker"],
                markersize=8, linewidth=2, capsize=4, label=m["short"], zorder=3)
ax.axhline(0, color="0.3", linestyle="--", linewidth=1)
ax.set_xticks(xpos, xlabels)
ax.set_xlabel("probabilitate depolarizantă p per poartă")
ax.set_title("Degradarea AUPRC față de propriul p = 0\n(linii subțiri: seeds)")
ax.legend(loc="lower left", fontsize=8)
for ax, p in zip(axes[1:], [0.01, 0.05]):
    order = ["A", "B", LR_KEY]
    for seed in SEEDS:
        vals = [part(k, p, "AUPRC")[seed] for k in order]
        ax.plot(range(3), vals, color=SEED_DOT_COLOR, marker="o", markersize=5, linewidth=1, alpha=0.8)
        ax.annotate(f"s{seed}", (2, vals[-1]), textcoords="offset points", xytext=(6, 0), fontsize=8, color="0.35", va="center")
    for i, k in enumerate(order):
        v = part(k, p, "AUPRC")
        ax.errorbar(i, v.mean(), yerr=v.std(ddof=1), color=MODELS[k]["color"] if k in MODELS else LR_COLOR, marker="D",
                    markersize=9, capsize=5, linewidth=2, zorder=3)
    ax.set_xticks(range(3), [SHORT[k] for k in order])
    ax.set_xlim(-0.4, 2.5)
    ax.set_title(f"AUPRC pe test per seed la p = {p}\n(linie gri = același seed în cele 3 modele)")
fig.suptitle("Robustețe și stabilitate între seeds, test", fontsize=11)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "degradation_and_stability.png", dpi=130)
plt.show()

# validation vs test
fig, ax = plt.subplots(figsize=(6, 5.2))
p_markers = dict(zip(P_ALL, ["o", "^", "s", "D"]))
for key in ["A", "B", LR_KEY]:
    color = MODELS[key]["color"] if key in MODELS else LR_COLOR
    for p in (P_ALL if key in MODELS else [0.0]):
        ax.scatter(part(key, p, "val AUPRC"), part(key, p, "AUPRC"), color=color, marker=p_markers[p], s=40, alpha=0.8,
                   label=f"{SHORT[key]}" + (f", p = {p}" if key in MODELS else ""))
lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
ax.plot(lims, lims, color="0.4", linestyle="--", linewidth=1)
ax.set_xlabel("AUPRC pe validation")
ax.set_ylabel("AUPRC pe test")
ax.set_title("Validation vs test, câte un punct per (model, seed, p)")
ax.legend(fontsize=7, ncol=2, loc="upper left")
plt.tight_layout()
fig.savefig(RESULTS_DIR / "validation_vs_test.png", dpi=130)
plt.show()

## 8. Cost ledger: antrenare, validare, test

Formula din Partea 8 (parameter shift, $1 + 2P$ execuții per exemplu de antrenare, $P = 32$; o execuție per tranzacție la scorare; 1024 shots per execuție). Fiecare execuție este numărată **o singură dată**, în experimentul în care a fost făcută prima oară:

| componentă | unde a fost executată |
|---|---|
| antrenare (30 de epoci × 590 de exemple × 65) | A: Exp. 1 · B: Exp. 3 |
| validare după fiecare epocă (30 × 56.962) | A: Exp. 1 · B: Exp. 3 |
| validation p = 0 cu ponderile alese | A: Exp. 1 · B: Exp. 3 |
| validation p = 0.001 / 0.01 / 0.05 | A: Exp. 2 · B: Exp. 3 |
| **test p = 0 / 0.001 / 0.01 / 0.05** | **Exp. 5 (acum)** |

Recalcularea scorurilor de validation din acest notebook (necesară ca să regăsim pragurile) este **timp de simulator, nu cost hardware nou**: pe un dispozitiv real scorurile de validation deja măsurate ar fi refolosite. Apare în ledger cu 0 execuții noi.

**Cost hardware ≠ timp de simulare.** Execuțiile și shots-urile sunt nominale (simularea rulează exact, fără shots). Timpul efectiv al simulatorului este într-un tabel separat. Pe un calculator cuantic real nu alegem p: zgomotul este al dispozitivului, deci o implementare reală ar plăti **un singur** nivel (o trecere pe validation fără p = 0 separat și o trecere pe test); cele 4 niveluri sunt scenarii simulate.

In [ ]:
n_val, n_test = len(y_val), len(y_test)
exp3_ledger = pd.read_csv(EXP3_DIR / "cost_ledger.csv", index_col=0)
exp3_structure = pd.read_csv(EXP3_DIR / "circuit_structure.csv", index_col=0)
exp1_runs = {r["seed"]: r for r in load_jsonl(EXP1_DIR / "runs.jsonl", required=True) if r["ratio"] == 1}
exp3_train = {r["seed"]: r for r in load_jsonl(EXP3_DIR / "train_runs.jsonl", required=True)}
assert all(exp1_runs[s]["epoci executate"] == EPOCHS and exp3_train[s]["epoci executate"] == EPOCHS for s in SEEDS)

# structura circuitului, recalculată pe ponderile reale și verificată față de Exp. 3
x_probe = torch.tensor(inputs[SEEDS[0]]["Z_test"][0])
structure_rows = {}
for key, m in MODELS.items():
    w = weights[key][SEEDS[0]]
    with use_feature_map_reps(m["reps"]):
        resources = qml.specs(qml.transforms.decompose(qml.QNode(vqc, device), gate_set=NATIVE_GATES))(x_probe, w)["resources"]
        structure_rows[m["label"]] = {"repetări ZZ": m["reps"], "porți native": resources.num_gates, "CNOT": resources.gate_types["CNOT"],
                                      "adâncime": resources.depth, "locuri de zgomot": noise_locations(vqc, x_probe, w), "parametri": w.numel()}
structure = pd.DataFrame(structure_rows).T
for (label, row), (_, ref) in zip(structure.iterrows(), exp3_structure.iterrows()):
    for col in ["porți native", "CNOT", "adâncime", "locuri de zgomot", "parametri"]:
        assert row[col] == ref[col], f"{label}: {col} = {row[col]}, în Exp. 3 {ref[col]}"
structure["execuții / exemplu antrenare (nominal 1 + 2P)"] = 1 + 2 * structure["parametri"]
structure["execuții / exemplu antrenare (Tracker, Exp. 3)"] = exp3_structure["execuții / exemplu (Tracker, parameter shift)"].to_numpy()
structure["execuții / tranzacție scorată"] = 1
structure.to_csv(RESULTS_DIR / "circuit_structure.csv")
EXEC_PER_SAMPLE = int(structure["execuții / exemplu antrenare (nominal 1 + 2P)"].iloc[0])
assert (structure["parametri"] == 32).all() and EXEC_PER_SAMPLE == 65

tested = {key: {s for (k, s, p) in test_done if k == key} for key in MODELS}
assert all(tested[key] == set(SEEDS) for key in MODELS)
n_s = len(SEEDS)
# (componentă, fază, executat în A, executat în B, execuții per seed)
ledger_rows = [
    ("1. antrenare: forward + parameter shift", "antrenare", "Exp. 1", "Exp. 3", EPOCHS * 590 * EXEC_PER_SAMPLE),
    ("2. validare după fiecare epocă, p = 0", "validare", "Exp. 1", "Exp. 3", EPOCHS * n_val),
    ("3a. validation p = 0, ponderile alese", "validare", "Exp. 1", "Exp. 3", n_val),
    ("3b. validation sub zgomot, p = 0.001 / 0.01 / 0.05", "validare", "Exp. 2", "Exp. 3", len(NOISE_LEVELS) * n_val),
    ("3c. recalculare validation în Exp. 5 (simulator, deja numărată la 3a / 3b)", "validare", "Exp. 5", "Exp. 5", 0),
    ("4. test, p = 0 / 0.001 / 0.01 / 0.05", "test", "Exp. 5", "Exp. 5", len(P_ALL) * n_test),
]
for (name, *_, per_seed_exec), (ref_name, ref) in zip([r for r in ledger_rows if not r[0].startswith("3c")], exp3_ledger.iterrows()):
    assert per_seed_exec == ref["execuții / seed, Exp. 3 (ZZ ×1)"] == ref["execuții / seed, baseline Exp. 2 (ZZ ×2)"], name

ledger = pd.DataFrame([
    {"componentă": name, "fază": phase, "executat în (A / B)": f"{ea} / {eb}", "execuții / seed (A și B)": per_seed_exec,
     f"execuții {n_s} seeds, A": per_seed_exec * n_s, f"execuții {n_s} seeds, B": per_seed_exec * n_s,
     f"shots {n_s} seeds, A": per_seed_exec * n_s * SHOTS_PER_EXECUTION, f"shots {n_s} seeds, B": per_seed_exec * n_s * SHOTS_PER_EXECUTION}
    for name, phase, ea, eb, per_seed_exec in ledger_rows
]).set_index("componentă")
lr_row = pd.DataFrame([{"componentă": "C. regresie logistică (antrenare, validare, test)", "fază": "toate", "executat în (A / B)": "CPU clasic",
                        "execuții / seed (A și B)": 0, f"execuții {n_s} seeds, A": 0, f"execuții {n_s} seeds, B": 0,
                        f"shots {n_s} seeds, A": 0, f"shots {n_s} seeds, B": 0}]).set_index("componentă")
ledger = pd.concat([ledger, lr_row])
ledger.to_csv(RESULTS_DIR / "cost_ledger.csv")

num_cols = ["execuții / seed (A și B)", f"execuții {n_s} seeds, A", f"execuții {n_s} seeds, B", f"shots {n_s} seeds, A", f"shots {n_s} seeds, B"]
totals = ledger.drop(index=lr_row.index).groupby("fază", sort=False)[num_cols].sum()
totals.loc["TOTAL"] = totals.sum()
totals.to_csv(RESULTS_DIR / "cost_totals.csv")

hardware_one_level = pd.DataFrame([
    {"scenariu": "simulat aici (4 niveluri de zgomot)", "validare evaluare / seed": (1 + len(NOISE_LEVELS)) * n_val, "test / seed": len(P_ALL) * n_test},
    {"scenariu": "un dispozitiv real (un singur nivel)", "validare evaluare / seed": n_val, "test / seed": n_test},
]).set_index("scenariu")
hardware_one_level["total / seed (cu antrenare + validare pe epoci)"] = hardware_one_level.sum(axis=1) + EPOCHS * 590 * EXEC_PER_SAMPLE + EPOCHS * n_val
hardware_one_level[f"shots {n_s} seeds"] = hardware_one_level["total / seed (cu antrenare + validare pe epoci)"] * n_s * SHOTS_PER_EXECUTION
hardware_one_level.to_csv(RESULTS_DIR / "cost_hardware_one_level.csv")

fmt = lambda v: f"{v:,}" if isinstance(v, (int, np.integer)) else v
print("Structura circuitului (determină zgomotul acumulat și durata unui shot, nu numărul de execuții):")
display(structure)
reduction = {k: 1 - structure.iloc[1][k] / structure.iloc[0][k] for k in ["porți native", "CNOT", "adâncime", "locuri de zgomot"]}
print("Reducere B față de A: " + ", ".join(f"{k} −{v:.0%}" for k, v in reduction.items()))
print(f"\nCost ledger (nominal, parameter shift, {SHOTS_PER_EXECUTION} shots/execuție), fiecare execuție numărată o dată:")
display(ledger.map(fmt))
print("Totaluri pe fază (A și B au același număr de execuții: aceiași 32 de parametri, aceleași date și epoci):")
display(totals.map(fmt))
print("Scenariu hardware real (un singur nivel de zgomot, al dispozitivului):")
display(hardware_one_level.map(fmt))

sim_time = pd.DataFrame([
    {"ce": f"Exp. 5: scorare val + test, model {key}", "timp simulator [min]": raw.loc[raw["model"] == key, "timp scorare val + test [s]"].sum() / 60}
    for key in [*MODELS, LR_KEY]
] + [
    {"ce": "antrenare A (Exp. 1)", "timp simulator [min]": sum(exp1_runs[s]["timp antrenare [s]"] for s in SEEDS) / 60},
    {"ce": "antrenare B (Exp. 3)", "timp simulator [min]": sum(exp3_train[s]["timp antrenare [s]"] for s in SEEDS) / 60},
]).set_index("ce")
sim_time.to_csv(RESULTS_DIR / "simulator_time.csv")
print("Timp efectiv de simulare (informativ, depinde de mașină; NU este cost hardware):")
display(sim_time.round(2))

## 9. Tabelul final (pentru prezentare)

Valori pe **test**, medie ± std pe 5 seeds. Metrica principală: AUPRC la p = 0.01 și p = 0.05.

In [ ]:
final_rows = []
for key in ["A", "B", LR_KEY]:
    is_vqc = key in MODELS
    r = {"model": LABELS[key]}
    for p in [0.0, 0.01, 0.05]:
        r[f"AUPRC p = {p}"] = mean_std(part(key, p, "AUPRC"))
    r["Spearman ρ p = 0.05"] = mean_std(part(key, 0.05, "Spearman rho")) if is_vqc else "—"
    r["F1 p = 0.05"] = mean_std(part(key, 0.05, "F1"))
    r["MCC p = 0.05"] = mean_std(part(key, 0.05, "MCC"))
    r["CNOT / locuri de zgomot"] = f"{int(structure.iloc[0 if key == 'A' else 1]['CNOT'])} / {int(structure.iloc[0 if key == 'A' else 1]['locuri de zgomot'])}" if is_vqc else "— (clasic)"
    r["execuții cuantice / seed"] = f"{int(totals.loc['TOTAL', 'execuții / seed (A și B)']):,}" if is_vqc else "0"
    final_rows.append(r)
final_table = pd.DataFrame(final_rows).set_index("model")
final_table.to_csv(RESULTS_DIR / "final_table.csv")
md_lines = ["| model | " + " | ".join(final_table.columns) + " |", "|---" * (len(final_table.columns) + 1) + "|"]
md_lines += ["| " + " | ".join([str(i), *map(str, r)]) + " |" for i, r in zip(final_table.index, final_table.to_numpy())]
(RESULTS_DIR / "final_table.md").write_text("\n".join(md_lines) + "\n", encoding="utf-8")   # fără tabulate (nu e în requirements)
display(final_table)

## 10. Cum citim rezultatele

- **AUPRC** (metrica principală) și **AUROC** depind doar de ordinea scorurilor; **Spearman ρ** arată cât din ordinea fără zgomot supraviețuiește. Precision / recall / F1 / MCC depind de prag; pe test pragul vine din validation la același p, deci sunt estimări oneste (spre deosebire de valorile de pe validation).
- Testul are 98 de fraude: mutarea unei singure fraude în clasament schimbă AUPRC vizibil. O diferență contează doar dacă este mare față de std-ul ei și are același semn la majoritatea seeds; 5 seeds nu permit un test statistic puternic.
- Diferențele A vs B sunt pereche (aceleași date, encoder și ponderi inițiale), deci singura cauză posibilă este circuitul.
- Regresia logistică nu are zgomot cuantic și costă 0 execuții cuantice: este reperul pe care un VQC trebuie să-l egaleze ca să fie util.
- Rezultatele de pe test nu au fost folosite pentru nicio alegere: modelele, ponderile, pragurile (din validation) și configurația erau fixate înainte de această rulare.

## 11. Concluzii (scrise după rularea finală; modelele nu au fost modificate în funcție de ele)

- **Cel mai mare AUPRC pe test: regresia logistică**, 0.703 ± 0.010, mai bună decât ambele VQC-uri la orice p și la fiecare seed (0/5 seeds cu B > C). Față de B: −0.064 la p = 0, −0.068 la p = 0.01, −0.086 la p = 0.05.
- **B (ZZ ×1) bate baseline-ul A (ZZ ×2)**: AUPRC +0.071 ± 0.059 la p = 0.01 (4/5 seeds) și +0.212 ± 0.183 la p = 0.05 (5/5 seeds). La p ≤ 0.01 seed 4 este o excepție (A puțin mai bun); la p = 0.05 câștigul este consecvent.
- **Robustețe:** de la p = 0 la p = 0.05, A pierde 0.176 AUPRC (−29%, un seed se prăbușește la 0.097), B pierde 0.022 (−3%). Spearman ρ la p = 0.05: A 0.687 ± 0.145, B 0.876 ± 0.109. F1 la p = 0.05: A 0.537 ± 0.184, B 0.691 ± 0.022.
- **Validation se confirmă:** AUPRC pe test este cu +0.01…+0.03 peste validation pentru ambele VQC-uri, iar avantajul B − A are aceeași mărime (p = 0.05: +0.206 pe validation, +0.212 pe test).
- **Cost:** același număr de execuții (3.315.056 per seed, aceiași 32 de parametri), dar circuitul B are −28% porți, −36% CNOT, −31% adâncime, −30% locuri de zgomot. Câștigul este în calitatea fiecărui shot, nu în numărul lor.